In [0]:
from pyspark.sql import functions as F

silver = spark.table("silver.retail_sales_clean")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# dim_payment
dim_payment = silver.select("payment").distinct() \
    .withColumn("payment_key", F.monotonically_increasing_id())
dim_payment.write.format("delta").mode("overwrite").saveAsTable("gold.dim_payment")

# dim_product
dim_product = silver.select("product_id").distinct() \
    .withColumn("product_key", F.monotonically_increasing_id())
dim_product.write.format("delta").mode("overwrite").saveAsTable("gold.dim_product")

# dim_customer
dim_customer = silver.select("customer_id").distinct() \
    .withColumn("customer_key", F.monotonically_increasing_id())
dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold.dim_customer")

# dim_date
dim_date = silver.select(F.to_date("transactional_date").alias("date")).distinct() \
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int")) \
    .withColumn("year", F.year("date")) \
    .withColumn("month", F.month("date")) \
    .withColumn("day", F.dayofmonth("date"))
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

print("Dimensions created")

Dimensions created


In [0]:
dim_payment = spark.table("gold.dim_payment")
dim_product = spark.table("gold.dim_product")
dim_customer = spark.table("gold.dim_customer")

fact_sales = (
    silver
    .join(dim_payment, silver["payment"] == dim_payment["payment"])
    .join(dim_product, silver["product_id"] == dim_product["product_id"])
    .join(dim_customer, silver["customer_id"] == dim_customer["customer_id"])
    .withColumn("date_key", F.date_format("transactional_date", "yyyyMMdd").cast("int"))
    .select(
        "transaction_id", "date_key", "payment_key", "product_key", "customer_key",
        "quantity", "cost", "price", "revenue", "margin", "loyalty_card"
    )
)

fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold.fact_sales")
print(f"Fact table rows: {fact_sales.count()}")

Fact table rows: 4234


In [0]:
display(spark.sql("""
    SELECT p.payment, COUNT(*) as txns, ROUND(SUM(f.revenue),2) as total_revenue
    FROM gold.fact_sales f JOIN gold.dim_payment p ON f.payment_key = p.payment_key
    GROUP BY p.payment ORDER BY total_revenue DESC
"""))

payment,txns,total_revenue
mastercard,1420,33656.35
americanexpress,1416,33642.11
visa,1398,32875.83


In [0]:
spark.sql("""
MERGE INTO gold.fact_sales AS target
USING silver.retail_sales_clean AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED THEN UPDATE SET target.price = source.price, target.revenue = source.price * source.quantity
WHEN NOT MATCHED THEN INSERT (transaction_id, quantity, cost, price)
VALUES (source.transaction_id, source.quantity, source.cost, source.price)
""")
print("MERGE completed")

MERGE completed


In [0]:
spark.sql("OPTIMIZE gold.fact_sales ZORDER BY (payment_key)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,